<a href="https://colab.research.google.com/github/sun-mengwei/dtb-colab-experiments/blob/codex%2Fgame-dynamics-dtb/parameter_evolving_neural_dtb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Parameter-evolving Neural--DTB: oscillatory deterministic game

This is a separate trial of the direct parameter update

\[
\theta_{k+1}=\theta_k+h\alpha_k,\qquad x_i^{k+1}=f_{\theta_{k+1}}(z_i).
\]

Unlike the fixed-chart/reset DTB notebook, the parameter Jacobian is recomputed at the new neural parameters every step. The target is the probability-flow velocity

\[
v_i^k=b(x_i^k)-\tfrac12Dq_i^k,
\]

and the log density and score are transported using the induced field \(u_k(f_{\theta_k}(z))=D_\theta f_{\theta_k}(z)\alpha_k\). The stacked SVD solve below is mathematically equivalent to solving \(G_k\alpha_k=P_k\), but avoids forming the less stable normal equations.

This trial uses the exact high-frequency two-player oscillatory drift from the deterministic notebook. It sets \(D=0\), samples the same uniform reference law on \([-1,1]^2\), and compares the parameter-evolving map with explicit Euler using the same particles and step size. The density and score equations are still advanced exactly as written in the algorithm.

In [ ]:
from pathlib import Path
import os, subprocess, sys, math, time
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.func import jacrev, vmap

REPO = Path("/content/dtb-colab-experiments")
BRANCH = "codex/game-dynamics-dtb"
if not (REPO / ".git").exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH,
                    "https://github.com/sun-mengwei/dtb-colab-experiments.git", str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "checkout", "-q", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "-q", "--ff-only"], check=True)
sys.path.insert(0, str(REPO / "DTB_rep"))
sys.path.insert(0, str(REPO / "dtb_game_dynamics_unnormalized"))
os.chdir(REPO)

try:
    import torch._dynamo.compiled_autograd
except (AttributeError, ImportError):
    pass
from dtb import device, flat_params, write_flat_into_model
from run_game_dtb import ResidualMLPMap, game_dtb_basis_matrices, map_at
from game_dtb.projection import truncated_svd_solve

# --------------------------- experiment controls ---------------------------
SEED = 2026
DIM = 2
N_DTB = 256
T, H = 1.0, 0.01
K = round(T / H)
WIDTH, DEPTH = 12, 1
SVD_RTOL = 1e-3  # direct theta updates activate near-null directions quickly
JACOBIAN_CHUNK = 128
DERIVATIVE_CHUNK = 32
DOMAIN_LOW, DOMAIN_HIGH = -1.0, 1.0
DIFFUSION_STRENGTH = 0.0        # exact deterministic game: D = 0
LAMBDA, GAMMA = 0.5, 0.2
EPSILON, OMEGA = 0.5, 4 * math.pi
SNAPSHOT_STEPS = sorted(set([0, K // 4, K // 2, 3 * K // 4, K]))

DEVICE = device()
DTYPE = torch.float32
torch.manual_seed(SEED)
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)
    torch.set_float32_matmul_precision("high")
D = DIFFUSION_STRENGTH * torch.eye(DIM, device=DEVICE, dtype=DTYPE)
print({"device": str(DEVICE), "steps": K, "h": H, "particles": N_DTB,
       "diffusion": DIFFUSION_STRENGTH})

In [ ]:
# High-frequency coupled game drift used in the deterministic notebook.
def b(x):
    x1, x2 = x.unbind(-1)
    return torch.stack((
        -LAMBDA * x1 - GAMMA * (x1 - x2) - EPSILON * torch.sin(OMEGA * x1),
        -LAMBDA * x2 - GAMMA * (x2 - x1) - EPSILON * torch.sin(OMEGA * x2),
    ), dim=-1)

def phi(x):
    x1, x2 = x.unbind(-1)
    return (
        -0.5 * LAMBDA * (x1.square() + x2.square())
        -0.5 * GAMMA * (x1 - x2).square()
        +(EPSILON / OMEGA) * (torch.cos(OMEGA * x1) + torch.cos(OMEGA * x2))
    )

def uniform_log_density_and_score(x):
    # rho_0 is constant in the box interior; sampled boundary points have
    # probability zero.  The classical score is not defined on the boundary.
    log_value = -DIM * math.log(DOMAIN_HIGH - DOMAIN_LOW)
    return torch.full((x.shape[0],), log_value, device=x.device, dtype=x.dtype), torch.zeros_like(x)

# Fixed labels z_i ~ Uniform([-1,1]^2).  The residual map begins exactly
# at identity, so these are also the initial physical particles.
z = (DOMAIN_LOW + (DOMAIN_HIGH - DOMAIN_LOW)
     * torch.rand(N_DTB, DIM, dtype=DTYPE)).to(DEVICE)
ell, q = uniform_log_density_and_score(z)
model = ResidualMLPMap(dim=DIM, width=WIDTH, depth=DEPTH,
                       activation="tanh", dtype=DTYPE).to(DEVICE)
theta, structure, _ = flat_params(model)
theta = theta.to(DEVICE)
selected = torch.arange(theta.numel(), device=DEVICE)  # full parameter evolution
x0 = map_at(theta, z, model, structure).detach()
assert torch.allclose(x0, z, atol=1e-7, rtol=0), "initial map must be identity"
print(f"trainable parameters: {theta.numel()} (all retained as candidates)")

In [ ]:
def tangent_spatial_terms(theta_k, alpha_k, labels):
    """Return u, grad_x u, div_x u, grad_x(div_x u), and map conditioning.

    The neural map is parameterized by z, whereas density transport needs
    derivatives in x.  At x=f_theta(z), the chain rule gives

        grad_x u = (D_z u) (D_z f_theta)^{-1}.

    Nested jacrev calls differentiate this expression once more to obtain
    grad_x(div u).  Smooth tanh activations are required here.
    """
    theta_constant = theta_k.detach().clone()
    alpha_constant = alpha_k.detach().clone()

    def map_one(theta_value, z_one):
        return map_at(theta_value, z_one.unsqueeze(0), model, structure).squeeze(0)

    parameter_jacobian = jacrev(map_one, argnums=0)

    def x_one(z_one):
        return map_one(theta_constant, z_one)

    def u_one(z_one):
        return parameter_jacobian(theta_constant, z_one) @ alpha_constant

    map_jacobian = jacrev(x_one)
    label_velocity_jacobian = jacrev(u_one)

    def grad_x_u_one(z_one):
        dx_dz = map_jacobian(z_one)
        du_dz = label_velocity_jacobian(z_one)
        return torch.linalg.solve(dx_dz.T, du_dz.T).T

    def divergence_one(z_one):
        return torch.trace(grad_x_u_one(z_one))

    divergence_gradient_z = jacrev(divergence_one)

    def grad_divergence_x_one(z_one):
        dx_dz = map_jacobian(z_one)
        grad_div_z = divergence_gradient_z(z_one)
        return torch.linalg.solve(dx_dz.T, grad_div_z)

    values = {name: [] for name in ("u", "grad_u", "div", "grad_div",
                                          "det", "condition")}
    for start in range(0, labels.shape[0], DERIVATIVE_CHUNK):
        batch = labels[start:start + DERIVATIVE_CHUNK]
        dx_dz = vmap(map_jacobian)(batch)
        values["u"].append(vmap(u_one)(batch))
        values["grad_u"].append(vmap(grad_x_u_one)(batch))
        values["div"].append(vmap(divergence_one)(batch))
        values["grad_div"].append(vmap(grad_divergence_x_one)(batch))
        values["det"].append(torch.linalg.det(dx_dz))
        values["condition"].append(torch.linalg.cond(dx_dz))
    return tuple(torch.cat(values[name]) for name in
                 ("u", "grad_u", "div", "grad_div", "det", "condition"))

In [ ]:
def parameter_evolving_dtb(theta_0, labels, ell_0, q_0):
    theta_k = theta_0.detach().clone()
    ell_k = ell_0.detach().clone()
    q_k = q_0.detach().clone()

    history = {
        "step": [], "particles": [], "trajectory": [],
        "log_density": [], "score": [],
        "projection_residual": [], "retained_rank": [],
        "sigma_max": [], "sigma_min_retained": [], "alpha_norm": [],
        "mean_divergence": [], "min_abs_map_det": [],
        "max_map_condition": [], "map_step_error": [],
    }

    def save_state(step, particles):
        history["step"].append(step)
        history["particles"].append(particles.detach().cpu())
        history["log_density"].append(ell_k.detach().cpu())
        history["score"].append(q_k.detach().cpu())

    x_k = map_at(theta_k, labels, model, structure).detach()
    save_state(0, x_k)
    history["trajectory"].append(x_k.detach().cpu())
    tic = time.perf_counter()

    for k in range(K):
        # J_i^k = D_theta f_theta(z_i), evaluated at the current parameters.
        x_map, jacobians, stacked_jacobian = game_dtb_basis_matrices(
            theta_k, selected, labels, model, structure, chunk=JACOBIAN_CHUNK
        )
        x_k = x_map.detach()

        # Probability-flow target v_i^k = b(x_i^k) - 1/2 D q_i^k.
        target_velocity = b(x_k) - 0.5 * (q_k @ D.T)
        alpha_k, svd = truncated_svd_solve(
            stacked_jacobian, target_velocity.reshape(-1), rtol=SVD_RTOL
        )

        # Spatial terms use the same old theta_k and alpha_k.
        u_k, grad_u, divergence, grad_divergence, map_det, map_condition = (
            tangent_spatial_terms(theta_k, alpha_k, labels)
        )
        transported_score = torch.einsum("nji,nj->ni", grad_u, q_k)
        ell_next = ell_k - H * divergence
        q_next = q_k - H * (transported_score + grad_divergence)

        # Direct parameter evolution: no frozen chart, accumulation, or refit.
        theta_next = theta_k + H * alpha_k
        x_next = map_at(theta_next, labels, model, structure).detach()
        euler_map_prediction = x_k + H * u_k.detach()
        map_step_error = torch.sqrt(
            torch.mean(torch.sum((x_next - euler_map_prediction).square(), dim=1))
        )

        history["projection_residual"].append(svd.relative_residual)
        history["retained_rank"].append(svd.retained_rank)
        history["sigma_max"].append(svd.sigma_max)
        history["sigma_min_retained"].append(svd.sigma_min_retained)
        history["alpha_norm"].append(float(torch.linalg.norm(alpha_k)))
        history["mean_divergence"].append(float(divergence.mean()))
        history["min_abs_map_det"].append(float(map_det.abs().min()))
        history["max_map_condition"].append(float(map_condition.max()))
        history["map_step_error"].append(float(map_step_error))

        history["trajectory"].append(x_next.detach().cpu())
        theta_k, ell_k, q_k = theta_next.detach(), ell_next.detach(), q_next.detach()
        if k + 1 in SNAPSHOT_STEPS:
            save_state(k + 1, x_next)
        if (k + 1) % max(1, K // 5) == 0:
            print(f"{k + 1:3d}/{K}: residual={svd.relative_residual:.2e}, "
                  f"rank={svd.retained_rank}, |alpha|={float(alpha_k.norm()):.2e}, "
                  f"min|det Dz f|={float(map_det.abs().min()):.2e}")

    write_flat_into_model(model, theta_k, structure)
    history["wall_seconds"] = time.perf_counter() - tic
    history["theta_K"] = theta_k.detach().cpu()
    history["X_K"] = map_at(theta_k, labels, model, structure).detach().cpu()
    history["trajectory"] = torch.stack(history["trajectory"])
    for key in ("projection_residual", "retained_rank", "sigma_max",
                "sigma_min_retained", "alpha_norm", "mean_divergence",
                "min_abs_map_det", "max_map_condition", "map_step_error"):
        history[key] = np.asarray(history[key])
    return theta_k, ell_k, q_k, history

In [ ]:
# Explicit Euler reference for the exact same fixed particles and h.
x_ref = z.detach().clone()
reference_trajectory = [x_ref.cpu()]
for _ in range(K):
    x_ref = x_ref + H * b(x_ref)
    reference_trajectory.append(x_ref.detach().cpu())
reference_trajectory = torch.stack(reference_trajectory)

theta_K, ell_K, q_K, result = parameter_evolving_dtb(theta, z, ell, q)
assert torch.isfinite(theta_K).all()
assert torch.isfinite(ell_K).all() and torch.isfinite(q_K).all()
assert result["X_K"].shape == (N_DTB, DIM)
assert result["trajectory"].shape == reference_trajectory.shape

trajectory_rmse = torch.sqrt(torch.mean(torch.sum(
    (result["trajectory"] - reference_trajectory).square(), dim=-1
), dim=1))
reference_rms = torch.sqrt(torch.mean(torch.sum(reference_trajectory.square(), dim=-1), dim=1))
relative_rmse = trajectory_rmse / (reference_rms + 1e-12)
potential_ref = phi(reference_trajectory).mean(1)
potential_dtb = phi(result["trajectory"]).mean(1)

print(f"wall time: {result['wall_seconds']:.1f} s")
print(f"mean/max projection residual: {result['projection_residual'].mean():.3e} / "
      f"{result['projection_residual'].max():.3e}")
print(f"final trajectory RMSE: {trajectory_rmse[-1]:.3e} "
      f"(relative {relative_rmse[-1]:.3e})")
print(f"final mean DTB={result['X_K'].mean(0).numpy()}, "
      f"Euler={reference_trajectory[-1].mean(0).numpy()}")

In [ ]:
# Same-particle comparison with explicit Euler for the oscillatory game.
grid_axis = torch.linspace(-1.05, 1.05, 140)
grid_x1, grid_x2 = torch.meshgrid(grid_axis, grid_axis, indexing="xy")
potential_grid = phi(torch.stack((grid_x1.ravel(), grid_x2.ravel()), -1)).reshape(grid_x1.shape)
fig, axes = plt.subplots(2, len(SNAPSHOT_STEPS), figsize=(3 * len(SNAPSHOT_STEPS), 6),
                         sharex=True, sharey=True)
for column, step in enumerate(SNAPSHOT_STEPS):
    dtb_index = result["step"].index(step)
    dtb_points = result["particles"][dtb_index].numpy()
    ref_points = reference_trajectory[step].numpy()
    for row in range(2):
        axes[row, column].contour(grid_x1, grid_x2, potential_grid, levels=16,
                                  linewidths=.45, alpha=.55)
    axes[0, column].scatter(dtb_points[:, 0], dtb_points[:, 1], s=8, alpha=.55)
    axes[1, column].scatter(ref_points[:, 0], ref_points[:, 1], s=8, alpha=.55)
    axes[0, column].set_title(f"t={step * H:.2f}")
    axes[0, column].grid(alpha=.2); axes[1, column].grid(alpha=.2)
    axes[0, column].set_aspect("equal"); axes[1, column].set_aspect("equal")
axes[0, 0].set_ylabel("parameter-evolving DTB")
axes[1, 0].set_ylabel("explicit Euler reference")
for axis in axes[1]: axis.set_xlabel(r"$x_1$")
plt.tight_layout(); plt.show()

state_times = np.arange(K + 1) * H
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].semilogy(state_times, trajectory_rmse + 1e-15, label="absolute")
axes[0].semilogy(state_times, relative_rmse + 1e-15, label="relative")
axes[0].set_title("error against same-step Euler"); axes[0].legend()
axes[1].plot(state_times, potential_ref, label="Euler")
axes[1].plot(state_times, potential_dtb, label="parameter DTB")
axes[1].set_title("mean potential"); axes[1].legend()
axes[2].plot(state_times, torch.linalg.vector_norm(
    result["trajectory"] - reference_trajectory, dim=-1).max(1).values)
axes[2].set_title("maximum particle error")
for axis in axes:
    axis.set_xlabel("time"); axis.grid(alpha=.25)
plt.tight_layout(); plt.show()

times = np.arange(K) * H
fig, axes = plt.subplots(2, 3, figsize=(15, 7.5))
axes[0, 0].semilogy(times, result["projection_residual"] + 1e-15)
axes[0, 0].set_title(r"projection residual $\|J\alpha-v\|/\|v\|$")
axes[0, 1].semilogy(times, result["alpha_norm"] + 1e-15)
axes[0, 1].set_title(r"$\|\alpha_k\|_2$")
axes[0, 2].semilogy(times, result["sigma_min_retained"] + 1e-15)
axes[0, 2].set_title(r"$\sigma_{\min,\mathrm{retained}}(J_k)$")
axes[1, 0].plot(times, result["retained_rank"])
axes[1, 0].set_title("retained tangent rank")
axes[1, 1].semilogy(times, result["min_abs_map_det"] + 1e-15, label=r"min $|\det D_zf|$")
axes[1, 1].semilogy(times, result["max_map_condition"] + 1e-15, label=r"max $\kappa(D_zf)$")
axes[1, 1].set_title("map invertibility diagnostics"); axes[1, 1].legend()
axes[1, 2].semilogy(times, result["map_step_error"] + 1e-15)
axes[1, 2].set_title(r"nonlinear map-step error")
for axis in axes.ravel():
    axis.set_xlabel("time"); axis.grid(alpha=.25)
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
snapshot_times = np.asarray(result["step"]) * H
mean_log_density = [values.mean().item() for values in result["log_density"]]
mean_score_norm = [torch.linalg.vector_norm(values, dim=1).mean().item()
                   for values in result["score"]]
axes[0].plot(snapshot_times, mean_log_density, marker="o")
axes[0].set_title(r"mean transported $\log\rho$")
axes[1].plot(snapshot_times, mean_score_norm, marker="o")
axes[1].set_title(r"mean transported $\|q\|_2$")
for axis in axes:
    axis.set_xlabel("time"); axis.grid(alpha=.25)
plt.tight_layout(); plt.show()

## What to check before trusting the trial

1. A small projection residual says the instantaneous probability-flow velocity lies near the current parameter tangent space.
2. A falling smallest retained singular value together with a growing \(\lVert\alpha_k\rVert\) indicates an ill-conditioned parameter update even when the residual is small.
3. The map must remain locally invertible. A determinant approaching zero or a rapidly growing map-Jacobian condition number makes the spatial score derivatives unreliable.
4. The nonlinear map-step error compares the actual update \(f_{\theta+h\alpha}(z)\) with its tangent prediction \(f_\theta(z)+hJ_\theta(z)\alpha\). It should decrease under step-size refinement.
5. Because \(D=0\), explicit Euler supplies a paired trajectory reference for every label. Repeat with smaller \(h\), more particles, and wider maps before attributing a discrepancy to parameter evolution itself.